# Projeto 1: Limpeza dataset ecommerce

## Importação das bibliotecas


In [ ]:
import pandas as pd
import numpy as np

## Importação dos dados

In [ ]:

importacao=pd.read_csv('vendas_ecommerce_dados_brutos.csv')
df=importacao.copy()

## Entendendo os dados


In [ ]:
df.info()

## Tratando colunas


In [ ]:
#Tratando colunas numéricas float

#Limpando os valores inválidos e mantendo somente os dados numéricos e os separadores de decimais e de milhar
for col in ['valor_total','preco_unitario']:
    df[col]=df[col].astype('str').str.replace(r'[^\d\.\,]','',regex=True).str.strip()

#lidando com as colunas numéricas float despadronizadas, algumas usavam ',' como separador de decimal e outras usavam '.' oq dificultava em criar uma solução geral para a coluna 

for col in ['valor_total','preco_unitario']:
    virgula= df[col].str.contains(r',\d{2}$',regex=True)
    df.loc[virgula,col] = (df.loc[virgula,col].str.replace('.','')
                            .str.replace(',','.')
                            .str.strip())
    

for col in ['valor_total','preco_unitario']:
    df.loc[df[col]=='', col] = np.nan
    df[col]=df[col].astype('float')


In [ ]:
# essas linhas servem para lidar com os caracteres que não são dígitos nas respectivas colunas
df['avaliacao_cliente']=df['avaliacao_cliente'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade']=df['quantidade'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade'].value_counts()


In [ ]:
#trocando os espaços por nulos para poder criar colunas numéricas e imputar valores
for col in ['quantidade','avaliacao_cliente']:
   df.loc[df[col]=='',col] = np.nan

# A conversão para int não funciona por haver nulos
df['quantidade']=df['quantidade'].astype('float')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('float')

#imputando a mediana aos nulos
for col in ['quantidade','preco_unitario','avaliacao_cliente']:
   df[col]=df[col].fillna((df[col].median(skipna=True)))

#conversão para int
df['quantidade']=df['quantidade'].astype('Int64')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('Int64')

In [ ]:
df['valor_total']=df['valor_total'].fillna((df['preco_unitario']*df['quantidade']))
df[['valor_total', 'preco_unitario', 'quantidade']].isna().sum()

In [ ]:

df['id_cliente']=df['id_cliente'].astype('str').str.replace(r'\D','',regex=True).str.strip()
df.loc[df['id_cliente']=='','id_cliente'] = np.nan

# Exclui registros sem id_cliente, pois sem esse identificador
# não é possível relacionar o cliente a consultas posteriores.
df = df.dropna(subset=['id_cliente'])

In [ ]:
#retirando não dígitos
df['idade']=df['idade'].str.replace(r'\D','',regex=True)
df['idade']=df['idade'].str.strip()
df.loc[df['idade']=='','idade'] = np.nan

# O int padrão python/numpy não aceita nulos, diferente do Int do pandas (o float padrão aceita também).
df['idade']=df['idade'].astype('Int64')


In [ ]:
#imputando nulos para a coluna de idade
# essas condição normalmente iria depender das regras de negócio, então
df.loc[df['idade']<18,'idade'] = df['idade'].median()
df['idade']=df['idade'].fillna((df['idade'].median()))
df.loc[df['idade']>80,'idade'] = df['idade'].median()


In [ ]:
#trocando os separadores das datas
df['data_pedido']=df['data_pedido'].str.replace('-','/')

#formatando a coluna para o tipo data
df['data_pedido']=pd.to_datetime(df['data_pedido'],format='mixed')

In [ ]:
#lidando com caracteres indesejados 
df['nome_cliente']=df['nome_cliente'].str.replace(r'[^\w\s\.]','',regex=True).str.title().str.strip()
df.loc[df['nome_cliente']=='','nome_cliente'] = np.nan

df.info()


In [ ]:

df['telefone']=df['telefone'].str.replace(r'[^\d\-]','',regex=True).str.replace(r'^55','',regex=True)
df['telefone']=df['telefone'].replace('-','') 
#sem o str o python n analisa os caracteres individualmente, mas sim o valor da célula como um todo o que faz o código não excluir o separador dos telefones
df.loc[df['telefone']=='','telefone'] = np.nan

for col in ['forma_pagamento','status_pedido','categoria_produto','cidade']:
    df[col]=df[col].astype('str').str.strip().str.title()
    df[col]=df[col].str.replace(r'[^\w\s]','',regex=True)
    df.loc[df[col]=='',col] = np.nan
    
df['produto']=df['produto'].astype('str').str.strip().str.title()
df['produto']=df['produto'].str.replace(r'[^\w\s]','',regex=True)
df.loc[df['produto']=='','produto'] = np.nan




In [ ]:
df['email']=df['email'].str.replace(r'[^\w\.\+\-\@]','',regex=True).str.lower()
df.loc[df['email']=='','email'] = np.nan

In [ ]:
#padronizando os registros de estado
df['estado'] = df['estado'].str.upper()

mapa_estados = {
    'MINAS GERAIS': 'MG',
    'CEARÁ': 'CE', 'CEARA': 'CE',
    'BAHIA': 'BA',
    'PERNAMBUCO': 'PE',
    'RIO GRANDE DO SUL': 'RS',
    'AMAZONAS': 'AM',
    'RIO DE JANEIRO': 'RJ',
    'PARANÁ': 'PR', 'PARANA': 'PR',
    'SANTA CATARINA': 'SC',
    'SÃO PAULO': 'SP', 'SAO PAULO': 'SP',
    'SERGIPE': 'SE',
    'DISTRITO FEDERAL': 'DF',
    'NÃO INFORMADO': np.nan, 'NAO INFORMADO': np.nan, '?': np.nan,
}

df['estado'] = df['estado'].replace(mapa_estados)
df

In [ ]:

df.duplicated().sum()
df=df.drop_duplicates()
df=df.fillna('Não Informado')
df.describe()

In [ ]:
df.isnull().sum()
df.duplicated().sum()

## Exportação


In [ ]:
df.to_csv('ecommerce_limpo.csv', index=False)